# 🔬 4주차 보강 — NumPy와 Pandas, "왜?"를 묻다

> **도구 사용법이 아니라, 도구가 존재하는 이유를 이해한다.**

---

## 이 노트북의 목표

| 기존 (도구 설명서 수준) | 보강 (전문가 수준 + 최신 AI 연결) |
|---|---|
| "`np.array()`로 배열을 만든다" | **왜** 리스트 대신 배열인가? → 메모리 구조 시각화 → **텐서**(Tensor)의 정체 |
| "`@`로 행렬 곱을 한다" | **왜** for문 대신 행렬 곱인가? → GPU 병렬 연산 → 딥러닝이 가능한 이유 |
| "**브로드캐스팅**(Broadcasting)이 편리하다" | **왜** 작동하는가? → 메모리 복사 없이 연산 → PyTorch도 동일 원리 |
| "`pd.read_csv()`로 데이터를 불러온다" | DataFrame이 **왜** AI/ML의 출발점인가? → **특성 공학**(Feature Engineering) |
| "`describe()`로 통계를 본다" | 이 통계량이 **왜** 중요한가? → **데이터 중심 AI**(Data-Centric AI) 패러다임 |

### 🔑 핵심 메시지

> *"NumPy 배열은 곧 텐서(Tensor)이다. `np.array()`를 이해하면 `torch.tensor()`를 이해한 것이다."*


---

## 1. 리스트(list) vs 배열(ndarray) — 메모리의 비밀

### 왜 파이썬 리스트로는 AI를 만들 수 없는가?

파이썬 리스트는 **"무엇이든 담을 수 있는 가방"**이다. 편리하지만 느리다. NumPy 배열은 **"같은 크기의 상자만 담는 선반"**이다. 제약이 있지만 **압도적으로 빠르다.**

그 이유는 **메모리 구조**에 있다.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import time
import sys
import os

# ── 한글 폰트 ──
if 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    os.system('apt-get -qq install -y fonts-nanum > /dev/null 2>&1')
    import matplotlib.font_manager as fm
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if os.path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

C = {
    'input': '#3b82f6', 'weight': '#f59e0b', 'bias': '#8b5cf6',
    'sum': '#ef4444', 'act': '#10b981', 'output': '#10b981',
    'bg': '#0f172a', 'card': '#1e293b', 'text': '#e2e8f0',
    'dim': '#94a3b8', 'muted': '#64748b', 'accent': '#60a5fa',
}

# ═══════════════════════════════════════════════════
# 메모리 구조 시각화: list vs ndarray
# ═══════════════════════════════════════════════════

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), facecolor=C['bg'])

# ── 왼쪽: 파이썬 리스트 ──
ax1.set_facecolor(C['card'])
ax1.set_xlim(0, 12)
ax1.set_ylim(0, 8)
ax1.axis('off')
ax1.set_title('파이썬 리스트 — 메모리가 흩어져 있다', color=C['sum'],
              fontsize=14, fontweight='bold', pad=12)

# 리스트 포인터 박스
list_data = [3.14, 42, "hello", True]
pointer_positions = [(1, 6), (3.5, 6), (6, 6), (8.5, 6)]
data_positions = [(0.5, 2.5), (5, 1), (9, 3.5), (3, 0.5)]

for i, ((px, py), (dx, dy), val) in enumerate(zip(pointer_positions, data_positions, list_data)):
    # 포인터 박스
    box = FancyBboxPatch((px-0.6, py-0.4), 1.2, 0.8,
                          boxstyle="round,pad=0.1",
                          facecolor=C['sum']+'33', edgecolor=C['sum'], lw=1.5)
    ax1.add_patch(box)
    ax1.text(px, py, f'ptr[{i}]', ha='center', va='center',
             color=C['sum'], fontsize=8, fontfamily='monospace')

    # 데이터 (흩어진 위치)
    dbox = FancyBboxPatch((dx-0.7, dy-0.35), 1.4, 0.7,
                           boxstyle="round,pad=0.1",
                           facecolor=C['bg'], edgecolor=C['dim'], lw=1.5)
    ax1.add_patch(dbox)
    ax1.text(dx, dy, str(val), ha='center', va='center',
             color=C['text'], fontsize=10, fontweight='bold', fontfamily='monospace')

    # 화살표 (포인터 → 데이터)
    ax1.annotate('', xy=(dx, dy+0.35), xytext=(px, py-0.4),
                arrowprops=dict(arrowstyle='->', color=C['muted'],
                               lw=1, connectionstyle="arc3,rad=0.2"))

ax1.text(6, 7.5, '각 요소가 메모리 곳곳에 흩어져 있다 → 느리다',
         ha='center', color=C['dim'], fontsize=10)

# ── 오른쪽: NumPy 배열 ──
ax2.set_facecolor(C['card'])
ax2.set_xlim(0, 12)
ax2.set_ylim(0, 8)
ax2.axis('off')
ax2.set_title('NumPy 배열 — 메모리가 연속으로 나열된다', color=C['act'],
              fontsize=14, fontweight='bold', pad=12)

arr_data = [3.14, 2.71, 1.41, 0.58]
for i, val in enumerate(arr_data):
    x = 1.5 + i * 2.3
    box = FancyBboxPatch((x-0.9, 3.5), 1.8, 1.2,
                          boxstyle="round,pad=0.1",
                          facecolor=C['act']+'22', edgecolor=C['act'], lw=2)
    ax2.add_patch(box)
    ax2.text(x, 4.1, f'{val}', ha='center', va='center',
             color=C['act'], fontsize=14, fontweight='bold', fontfamily='monospace')
    ax2.text(x, 3.7, f'float64', ha='center', va='center',
             color=C['muted'], fontsize=8, fontfamily='monospace')

    if i < len(arr_data) - 1:
        ax2.annotate('', xy=(x+1.1, 4.1), xytext=(x+0.9, 4.1),
                    arrowprops=dict(arrowstyle='-', color=C['act'], lw=2))

ax2.text(6, 6.5, '모든 요소가 같은 타입, 같은 크기로 연속 배치',
         ha='center', color=C['act'], fontsize=11, fontweight='bold')
ax2.text(6, 5.8, '→ CPU/GPU가 한 번에 쭉 읽을 수 있다 → 빠르다!',
         ha='center', color=C['accent'], fontsize=11)

# 하단 비교
ax2.text(6, 1.5, '🔗 AI 연결: PyTorch의 torch.tensor()도\n'
         '    NumPy와 완전히 같은 연속 메모리 구조를 사용한다.\n'
         '    np.array → torch.tensor 변환이 거의 비용 없이 가능한 이유.',
         ha='center', va='center', color=C['accent'], fontsize=10,
         bbox=dict(facecolor=C['accent']+'11', edgecolor=C['accent']+'44',
                  boxstyle='round,pad=0.5'))

plt.tight_layout()
plt.show()

# ── 실측 비교 ──
n = 1_000_000
py_list = list(range(n))
np_arr = np.arange(n)

# 메모리 크기
list_size = sys.getsizeof(py_list) + sum(sys.getsizeof(x) for x in py_list[:100]) / 100 * n
arr_size = np_arr.nbytes

print(f"\n{'═'*55}")
print(f"  📊 리스트 vs 배열 비교 (원소 {n:,}개)")
print(f"{'═'*55}")
print(f"  메모리: 리스트 ≈ {list_size/1e6:.1f} MB  |  배열 = {arr_size/1e6:.1f} MB  |  {list_size/arr_size:.0f}배 차이")

# 속도
start = time.perf_counter()
result_list = [x * 2 for x in py_list]
t_list = time.perf_counter() - start

start = time.perf_counter()
result_arr = np_arr * 2
t_arr = time.perf_counter() - start

print(f"  속도:   리스트 = {t_list*1000:.1f}ms  |  배열 = {t_arr*1000:.2f}ms  |  {t_list/t_arr:.0f}배 빠름")
print(f"{'═'*55}")
print(f"\n💡 이 속도 차이가 100만 개에서 이 정도이면,")
print(f"   GPT-4의 1조 8000억 개 파라미터에서는 어떻겠는가?")
print(f"   → 리스트로는 물리적으로 불가능. NumPy/PyTorch 없이 AI는 없다.")


---

## 2. 텐서(Tensor)의 정체 — NumPy 배열의 다른 이름

**텐서**(Tensor)란 무엇인가? 겁먹을 필요 없다. 이미 사용하고 있었다.

| 차원 | NumPy | PyTorch | 예시 |
|:---:|---|---|---|
| 0차원 | `np.array(42)` | `torch.tensor(42)` | 하나의 숫자 (스칼라) |
| 1차원 | `np.array([1,2,3])` | `torch.tensor([1,2,3])` | 벡터, 오디오 신호 |
| 2차원 | `np.array([[1,2],[3,4]])` | `torch.tensor(...)` | 행렬, 흑백 이미지 |
| 3차원 | `np.zeros((3,224,224))` | `torch.zeros(...)` | 컬러 이미지 (RGB) |
| 4차원 | `np.zeros((32,3,224,224))` | `torch.zeros(...)` | 이미지 배치 (Batch) |

> **"NumPy의 ndarray를 이해하면, PyTorch의 Tensor를 이해한 것이다."**


In [ ]:
# ═══════════════════════════════════════════════════
# 텐서 차원 시각화
# ═══════════════════════════════════════════════════

fig, axes = plt.subplots(1, 4, figsize=(16, 4), facecolor=C['bg'])

# 0D: 스칼라
ax = axes[0]
ax.set_facecolor(C['card'])
ax.axis('off')
ax.set_title('0D: 스칼라(Scalar)', color=C['input'], fontsize=12, fontweight='bold')
circle = plt.Circle((0.5, 0.5), 0.15, color=C['input'], alpha=0.3,
                      transform=ax.transAxes)
ax.add_patch(circle)
ax.text(0.5, 0.5, '42', transform=ax.transAxes, ha='center', va='center',
        color=C['input'], fontsize=20, fontweight='bold', fontfamily='monospace')
ax.text(0.5, 0.12, 'shape: ()', transform=ax.transAxes, ha='center',
        color=C['dim'], fontsize=10, fontfamily='monospace')

# 1D: 벡터
ax = axes[1]
ax.set_facecolor(C['card'])
ax.axis('off')
ax.set_title('1D: 벡터(Vector)', color=C['act'], fontsize=12, fontweight='bold')
for i, v in enumerate([1.0, 2.0, 3.0, 4.0]):
    x = 0.15 + i * 0.2
    box = FancyBboxPatch((x-0.06, 0.4), 0.12, 0.2,
                          boxstyle="round,pad=0.02",
                          facecolor=C['act']+'33', edgecolor=C['act'], lw=1.5,
                          transform=ax.transAxes)
    ax.add_patch(box)
    ax.text(x, 0.5, f'{v:.0f}', transform=ax.transAxes, ha='center', va='center',
            color=C['act'], fontsize=11, fontweight='bold', fontfamily='monospace')
ax.text(0.5, 0.12, 'shape: (4,)', transform=ax.transAxes, ha='center',
        color=C['dim'], fontsize=10, fontfamily='monospace')

# 2D: 행렬
ax = axes[2]
ax.set_facecolor(C['card'])
ax.axis('off')
ax.set_title('2D: 행렬(Matrix)', color=C['weight'], fontsize=12, fontweight='bold')
matrix = [[1,2,3],[4,5,6]]
for i, row in enumerate(matrix):
    for j, v in enumerate(row):
        x = 0.22 + j * 0.2
        y = 0.58 - i * 0.2
        box = FancyBboxPatch((x-0.06, y-0.06), 0.12, 0.12,
                              boxstyle="round,pad=0.01",
                              facecolor=C['weight']+'33', edgecolor=C['weight'], lw=1,
                              transform=ax.transAxes)
        ax.add_patch(box)
        ax.text(x, y, str(v), transform=ax.transAxes, ha='center', va='center',
                color=C['weight'], fontsize=10, fontweight='bold', fontfamily='monospace')
ax.text(0.5, 0.12, 'shape: (2, 3)', transform=ax.transAxes, ha='center',
        color=C['dim'], fontsize=10, fontfamily='monospace')

# 3D: 텐서 (이미지)
ax = axes[3]
ax.set_facecolor(C['card'])
ax.axis('off')
ax.set_title('3D: 컬러 이미지', color=C['accent'], fontsize=12, fontweight='bold')
# 3개 채널을 겹친 형태로 표현
colors_rgb = ['#ef4444', '#10b981', '#3b82f6']  # R, G, B
labels_rgb = ['R', 'G', 'B']
for k in range(3):
    offset = k * 0.06
    box = FancyBboxPatch((0.2+offset, 0.3+offset), 0.4, 0.35,
                          boxstyle="round,pad=0.02",
                          facecolor=colors_rgb[k]+'33',
                          edgecolor=colors_rgb[k], lw=1.5,
                          transform=ax.transAxes)
    ax.add_patch(box)
    ax.text(0.2+offset+0.4, 0.3+offset+0.35, labels_rgb[k],
            transform=ax.transAxes, ha='center', va='center',
            color=colors_rgb[k], fontsize=9, fontweight='bold')
ax.text(0.5, 0.12, 'shape: (3, 224, 224)', transform=ax.transAxes,
        ha='center', color=C['dim'], fontsize=10, fontfamily='monospace')

plt.suptitle('텐서(Tensor) = 다차원 배열 — NumPy ndarray와 같은 것이다',
             color=C['text'], fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# NumPy ↔ PyTorch 비교 코드
print("\n🐍 NumPy ↔ PyTorch — 문법이 거의 같다:")
print("─" * 55)
print(f"  {'NumPy':<30} {'PyTorch':<30}")
print("─" * 55)
comparisons = [
    ("np.array([1,2,3])",       "torch.tensor([1,2,3])"),
    ("np.zeros((3,3))",         "torch.zeros((3,3))"),
    ("a @ b",                   "a @ b  (동일!)"),
    ("np.max(a)",               "torch.max(a)"),
    ("a.reshape(2,3)",          "a.view(2,3)"),
    ("a.T",                     "a.T  (동일!)"),
]
for np_code, pt_code in comparisons:
    print(f"  {np_code:<30} {pt_code:<30}")
print("─" * 55)
print("\n💡 NumPy를 배우면 PyTorch의 80%를 이미 아는 것이다.")


---

## 3. 벡터화 연산(Vectorization) — "왜 for문을 쓰면 안 되는가?"

### 핵심 질문

> *"100만 개의 숫자에 2를 곱하고 싶다. for문과 NumPy 중 뭘 써야 하는가?"*

답은 당연히 NumPy이다. 하지만 **왜?** 그 이유를 시각화로 이해한다.

### 두 가지 방식의 차이

```
for문:    하나씩 꺼내고 → 곱하고 → 다시 넣고 → 다음 꺼내고 → ...  (직렬)
NumPy:   전체를 한 번에 곱한다                                    (병렬)
```

이것은 단순히 "편리함"의 문제가 아니다. **물리적으로 다른 연산 방식**이다.


In [ ]:
# ═══════════════════════════════════════════════════
# 벡터화 vs for문 — 시각적 비교
# ═══════════════════════════════════════════════════

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), facecolor=C['bg'])

# ── 위: for문 (직렬) ──
ax1.set_facecolor(C['card'])
ax1.axis('off')
ax1.set_title('for문: 하나씩 순서대로 (직렬 처리)', color=C['sum'],
              fontsize=14, fontweight='bold', pad=12)

for i in range(8):
    x = 0.8 + i * 1.3
    alpha = 1.0 if i == 3 else 0.3  # 현재 처리 중인 것 강조

    # 입력
    box = FancyBboxPatch((x-0.4, 2.5), 0.8, 0.8,
                          boxstyle="round,pad=0.08",
                          facecolor=C['input']+'33' if alpha > 0.5 else C['bg'],
                          edgecolor=C['input'], lw=1.5, alpha=alpha)
    ax1.add_patch(box)
    ax1.text(x, 2.9, f'x[{i}]', ha='center', va='center',
             color=C['input'], fontsize=9, fontfamily='monospace', alpha=alpha)

    # 화살표
    if alpha > 0.5:
        ax1.annotate('', xy=(x, 1.8), xytext=(x, 2.5),
                    arrowprops=dict(arrowstyle='->', color=C['weight'], lw=2))
        ax1.text(x+0.5, 2.15, '×2', color=C['weight'],
                 fontsize=11, fontweight='bold')

    # 출력
    box2 = FancyBboxPatch((x-0.4, 0.8), 0.8, 0.8,
                           boxstyle="round,pad=0.08",
                           facecolor=C['act']+'33' if i <= 3 else C['bg'],
                           edgecolor=C['act'] if i <= 3 else C['muted'],
                           lw=1.5, alpha=1.0 if i <= 3 else 0.3)
    ax1.add_patch(box2)
    status = '✓' if i < 3 else ('→' if i == 3 else '?')
    ax1.text(x, 1.2, status, ha='center', va='center',
             color=C['act'] if i <= 3 else C['muted'], fontsize=12)

ax1.text(12, 2, '시간: O(n)
한 번에 하나만
처리 가능',
         ha='center', va='center', color=C['dim'], fontsize=10)
ax1.set_xlim(0, 13)
ax1.set_ylim(0, 4)

# ── 아래: NumPy (병렬) ──
ax2.set_facecolor(C['card'])
ax2.axis('off')
ax2.set_title('NumPy: 전체를 한 번에 (병렬 처리 = SIMD)', color=C['act'],
              fontsize=14, fontweight='bold', pad=12)

for i in range(8):
    x = 0.8 + i * 1.3

    # 입력 (모두 활성)
    box = FancyBboxPatch((x-0.4, 2.5), 0.8, 0.8,
                          boxstyle="round,pad=0.08",
                          facecolor=C['input']+'33',
                          edgecolor=C['input'], lw=1.5)
    ax2.add_patch(box)
    ax2.text(x, 2.9, f'x[{i}]', ha='center', va='center',
             color=C['input'], fontsize=9, fontfamily='monospace')

    # 화살표 (모두 동시)
    ax2.annotate('', xy=(x, 1.8), xytext=(x, 2.5),
                arrowprops=dict(arrowstyle='->', color=C['weight'], lw=2))

    # 출력 (모두 완료)
    box2 = FancyBboxPatch((x-0.4, 0.8), 0.8, 0.8,
                           boxstyle="round,pad=0.08",
                           facecolor=C['act']+'33',
                           edgecolor=C['act'], lw=1.5)
    ax2.add_patch(box2)
    ax2.text(x, 1.2, '✓', ha='center', va='center',
             color=C['act'], fontsize=12)

# ×2 전체에 걸친 표시
ax2.text(5.5, 2.15, '전체 × 2  (한 번에!)', ha='center', va='center',
         color=C['weight'], fontsize=13, fontweight='bold',
         bbox=dict(facecolor=C['weight']+'22', edgecolor=C['weight'],
                  boxstyle='round,pad=0.3'))
ax2.text(12, 2, '시간: O(1)\n(벡터 단위)\nCPU SIMD 명령어\n→ GPU는 수천 개 병렬',
         ha='center', va='center', color=C['accent'], fontsize=10)
ax2.set_xlim(0, 13)
ax2.set_ylim(0, 4)

plt.tight_layout()
plt.show()

print("\n🔗 AI 연결:")
print("   SIMD(Single Instruction, Multiple Data) → GPU의 CUDA 코어 → TPU")
print("   NumPy는 CPU에서 SIMD를 활용하고, PyTorch는 GPU의 수천 개 코어를 활용한다.")
print("   원리는 동일: '같은 연산을 여러 데이터에 동시 적용'")


---

## 4. 브로드캐스팅(Broadcasting) — "크기가 다른데 어떻게 연산이 되는가?"

### 핵심 질문

```python
a = np.array([[1, 2, 3],
              [4, 5, 6]])    # shape: (2, 3)

b = np.array([10, 20, 30])   # shape: (3,)

a + b = ?   # shape가 다른데 어떻게 더해지는가?
```

NumPy가 **자동으로 `b`를 복제하여 `a`와 같은 크기로 맞춰 준다.** 이것이 **브로드캐스팅**(Broadcasting)이다.

> **🔗 AI 연결**: PyTorch에서 **편향(bias) 덧셈**, **배치 정규화(Batch Normalization)**, **어텐션 마스크(Attention Mask)** 적용이 모두 브로드캐스팅으로 작동한다.


In [ ]:
# ═══════════════════════════════════════════════════
# 브로드캐스팅 시각화
# ═══════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=C['bg'])

a = np.array([[1, 2, 3], [4, 5, 6]])
b_vec = np.array([10, 20, 30])
result = a + b_vec

def draw_matrix(ax, data, x0, y0, color, cell_w=1.0, cell_h=0.7, title='',
                fontsize=13):
    rows = data if isinstance(data, list) else data.tolist()
    if not isinstance(rows[0], list):
        rows = [rows]
    for i, row in enumerate(rows):
        for j, val in enumerate(row):
            x = x0 + j * cell_w
            y = y0 - i * cell_h
            box = FancyBboxPatch((x-cell_w/2+0.05, y-cell_h/2+0.05),
                                  cell_w-0.1, cell_h-0.1,
                                  boxstyle="round,pad=0.05",
                                  facecolor=color+'22', edgecolor=color, lw=1.5)
            ax.add_patch(box)
            ax.text(x, y, str(val), ha='center', va='center',
                    color=color, fontsize=fontsize, fontweight='bold',
                    fontfamily='monospace')
    if title:
        ax.text(x0 + (len(rows[0])-1)*cell_w/2, y0 + cell_h + 0.1, title,
                ha='center', va='center', color=color, fontsize=10, fontweight='bold')

# ── Step 1: 원본 ──
ax = axes[0]
ax.set_facecolor(C['card'])
ax.axis('off')
ax.set_xlim(-0.5, 4)
ax.set_ylim(-2, 3)
ax.set_title('① 원래 모양', color=C['text'], fontsize=13, fontweight='bold')

draw_matrix(ax, a.tolist(), 1, 2, C['input'], title='A: shape(2,3)')
draw_matrix(ax, [b_vec.tolist()], 1, -0.5, C['weight'], title='b: shape(3,)')

# ── Step 2: 브로드캐스트 ──
ax = axes[1]
ax.set_facecolor(C['card'])
ax.axis('off')
ax.set_xlim(-0.5, 4)
ax.set_ylim(-2, 3)
ax.set_title('② 자동 확장 (복사!)', color=C['text'], fontsize=13, fontweight='bold')

draw_matrix(ax, a.tolist(), 1, 2, C['input'], title='A: shape(2,3)')
b_broadcast = np.broadcast_to(b_vec, (2, 3))
draw_matrix(ax, b_broadcast.tolist(), 1, -0.5, C['weight'], title='b → shape(2,3)')
ax.text(3.5, -0.5, '자동\n복제!', ha='center', va='center',
        color=C['accent'], fontsize=10, fontweight='bold')

# ── Step 3: 결과 ──
ax = axes[2]
ax.set_facecolor(C['card'])
ax.axis('off')
ax.set_xlim(-0.5, 4)
ax.set_ylim(-2, 3)
ax.set_title('③ 요소별 덧셈 결과', color=C['text'], fontsize=13, fontweight='bold')

draw_matrix(ax, result.tolist(), 1, 1, C['act'], title='A + b: shape(2,3)')
ax.text(2, -1, 'A + b 완료!\n메모리 복사 없이 연산!',
        ha='center', va='center', color=C['act'], fontsize=11, fontweight='bold')

plt.suptitle('브로드캐스팅(Broadcasting) — 크기가 다른 배열의 연산',
             color=C['text'], fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# 실제 코드
print("\n🐍 코드:")
print(f"   a = np.array({a.tolist()})")
print(f"   b = np.array({b_vec.tolist()})")
print(f"   a + b = {result.tolist()}")
print(f"\n🔗 AI 연결:")
print(f"   신경망에서 z = X @ W + b 할 때:")
print(f"   X @ W 결과 shape(배치, 출력)에 b shape(출력,)을 더하면")
print(f"   → 브로드캐스팅이 자동으로 b를 배치 수만큼 복제한다.")
print(f"   → 이것이 '편향을 모든 샘플에 동시 적용'하는 원리이다.")


---

## 5. DataFrame — "왜 AI/ML의 출발점인가?"

Pandas의 DataFrame은 단순한 표가 아니다. ML에서 DataFrame은 곧 **특성 행렬**(Feature Matrix)이다.

| DataFrame 용어 | ML 용어 | 설명 |
|---|---|---|
| 행(row) | **샘플**(Sample) | 하나의 데이터 포인트 |
| 열(column) | **특성**(Feature) | 모델이 학습하는 변수 |
| 특정 열 | **타겟**(Target, y) | 모델이 예측해야 할 값 |
| `df.values` | **X 행렬** | NumPy 배열로 변환 → 모델 입력 |

> **🔗 AI 연결**: Scikit-learn의 `model.fit(X, y)`에서 X는 DataFrame에서 나온다. ChatGPT도 텍스트를 숫자 행렬(임베딩)로 바꾼 뒤 동일한 구조로 학습한다.


In [ ]:
# ═══════════════════════════════════════════════════
# DataFrame → ML Feature Matrix 변환 시각화
# ═══════════════════════════════════════════════════
import pandas as pd

# 예시 데이터
data = {
    'age': [25, 45, 35, 50, 23],
    'income': [30000, 75000, 50000, 90000, 25000],
    'bought': [0, 1, 1, 1, 0]  # 타겟
}
df = pd.DataFrame(data)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=C['bg'])

# ── DataFrame ──
ax = axes[0]
ax.set_facecolor(C['card'])
ax.axis('off')
ax.set_xlim(0, 6)
ax.set_ylim(0, 8)
ax.set_title('① DataFrame (원본 표)', color=C['input'], fontsize=13, fontweight='bold')

cols = ['age', 'income', 'bought']
col_colors = [C['input'], C['input'], C['act']]
for j, (col, cc) in enumerate(zip(cols, col_colors)):
    ax.text(1 + j * 1.8, 7, col, ha='center', va='center',
            color=cc, fontsize=11, fontweight='bold', fontfamily='monospace')
    ax.plot([0.2 + j*1.8, 1.8 + j*1.8], [6.6, 6.6], color=cc, lw=1)
    for i in range(5):
        val = df.iloc[i][col]
        ax.text(1 + j * 1.8, 6 - i * 0.9, str(val), ha='center', va='center',
                color=C['text'], fontsize=10, fontfamily='monospace')

# ── 분리 ──
ax = axes[1]
ax.set_facecolor(C['card'])
ax.axis('off')
ax.set_xlim(0, 6)
ax.set_ylim(0, 8)
ax.set_title('② 특성(X)과 타겟(y) 분리', color=C['weight'], fontsize=13, fontweight='bold')

# X 박스
xbox = FancyBboxPatch((0.3, 1.5), 2.5, 5.5,
                        boxstyle="round,pad=0.2",
                        facecolor=C['input']+'11',
                        edgecolor=C['input'], lw=2)
ax.add_patch(xbox)
ax.text(1.55, 7.3, 'X (특성 행렬)', ha='center', color=C['input'],
        fontsize=11, fontweight='bold')

for j, col in enumerate(['age', 'income']):
    ax.text(0.9 + j * 1.3, 6.5, col, ha='center', color=C['input'],
            fontsize=9, fontweight='bold', fontfamily='monospace')
    for i in range(5):
        ax.text(0.9 + j * 1.3, 5.8 - i * 0.9, str(df.iloc[i][col]),
                ha='center', color=C['text'], fontsize=10, fontfamily='monospace')

# y 박스
ybox = FancyBboxPatch((3.5, 1.5), 1.5, 5.5,
                        boxstyle="round,pad=0.2",
                        facecolor=C['act']+'11',
                        edgecolor=C['act'], lw=2)
ax.add_patch(ybox)
ax.text(4.25, 7.3, 'y (타겟)', ha='center', color=C['act'],
        fontsize=11, fontweight='bold')
for i in range(5):
    ax.text(4.25, 5.8 - i * 0.9, str(df.iloc[i]['bought']),
            ha='center', color=C['act'], fontsize=10, fontfamily='monospace')

# ── ML 모델 입력 ──
ax = axes[2]
ax.set_facecolor(C['card'])
ax.axis('off')
ax.set_xlim(0, 6)
ax.set_ylim(0, 8)
ax.set_title('③ 모델 학습', color=C['accent'], fontsize=13, fontweight='bold')

# X → 모델 → y
ax.text(1.5, 6.5, 'X (NumPy 배열)', ha='center', color=C['input'],
        fontsize=11, fontweight='bold',
        bbox=dict(facecolor=C['input']+'22', edgecolor=C['input'],
                 boxstyle='round,pad=0.3'))
ax.annotate('', xy=(3, 5), xytext=(1.5, 6),
           arrowprops=dict(arrowstyle='->', color=C['dim'], lw=2))

# 모델 박스
model_box = FancyBboxPatch((1.5, 3.5), 3, 1.2,
                             boxstyle="round,pad=0.15",
                             facecolor=C['accent']+'22',
                             edgecolor=C['accent'], lw=2)
ax.add_patch(model_box)
ax.text(3, 4.1, 'model.fit(X, y)', ha='center', va='center',
        color=C['accent'], fontsize=13, fontweight='bold', fontfamily='monospace')

ax.annotate('', xy=(3, 3.5), xytext=(4.5, 5),
           arrowprops=dict(arrowstyle='<-', color=C['dim'], lw=2))
ax.text(4.5, 6.5, 'y (타겟)', ha='center', color=C['act'],
        fontsize=11, fontweight='bold',
        bbox=dict(facecolor=C['act']+'22', edgecolor=C['act'],
                 boxstyle='round,pad=0.3'))

# 코드
code_lines = [
    'X = df[["age","income"]].values',
    'y = df["bought"].values',
    'model.fit(X, y)',
    'model.predict(X_new)',
]
for i, line in enumerate(code_lines):
    ax.text(3, 2.5 - i * 0.5, line, ha='center', va='center',
            color=C['accent'], fontsize=9, fontfamily='monospace')

plt.tight_layout()
plt.show()

# 실제 변환
X = df[['age', 'income']].values
y = df['bought'].values
print(f"\n🐍 실제 변환:")
print(f"   X = df[['age', 'income']].values")
print(f"   X.shape = {X.shape}  ← (샘플 수, 특성 수)")
print(f"   y = df['bought'].values")
print(f"   y.shape = {y.shape}  ← (샘플 수,)")
print(f"\n   이제 model.fit(X, y)를 호출하면 학습이 시작된다!")
print(f"\n🔗 AI 연결:")
print(f"   ChatGPT도 동일한 구조: 텍스트 → 토큰 → 숫자 배열(임베딩) → 모델 입력")
print(f"   '사과' → [0.12, -0.34, 0.78, ...] (768차원 벡터)")
print(f"   결국 모든 AI는 숫자 배열(텐서)을 입력받아 숫자를 출력한다.")


---

## 6. `describe()` — "왜 통계량을 먼저 봐야 하는가?"

### 데이터 중심 AI(Data-Centric AI) 패러다임

2021년 **앤드류 응**(Andrew Ng)은 AI 커뮤니티에 중요한 전환을 선언했다:

> *"모델 중심(Model-Centric)에서 데이터 중심(Data-Centric)으로 전환하라.*
> *모델을 바꾸기보다 데이터의 품질을 높이는 것이 더 효과적이다."*

`describe()`는 이 "데이터 품질 점검"의 **첫 번째 도구**이다.

| 통계량 | 점검 목적 | 문제 발견 예시 |
|---|---|---|
| **count** | 결측값 존재 여부 | count가 다른 열보다 적으면 결측 |
| **mean vs median** | 이상치(Outlier) 존재 | mean >> median이면 극단값 존재 |
| **std** | 스케일 차이 | std가 열마다 수십~수백 배 다르면 스케일링 필요 |
| **min, max** | 데이터 범위 오류 | 나이가 -5 또는 300이면 오류 |
| **25%, 75%** | 분포 형태 | 사분위 범위(IQR) → 박스플롯의 근거 |


In [ ]:
# ═══════════════════════════════════════════════════
# describe()가 발견하는 문제들 — 시각화
# ═══════════════════════════════════════════════════

# 문제가 숨어 있는 데이터
np.random.seed(42)
n = 200
df_problem = pd.DataFrame({
    'age': np.concatenate([np.random.normal(35, 10, n-3), [150, -5, 999]]),
    'salary': np.concatenate([np.random.normal(50000, 15000, n-2), [5000000, -100]]),
    'score': np.concatenate([np.random.normal(75, 10, n-5),
                             [np.nan, np.nan, np.nan, np.nan, np.nan]]),
})

desc = df_problem.describe()

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=C['bg'])

problems_found = {
    'age': {
        'issue': '이상치(Outlier)',
        'detail': f'min={desc["age"]["min"]:.0f}, max={desc["age"]["max"]:.0f}\n나이가 -5, 150, 999?',
        'color': C['sum'],
        'mean_median': f'mean={desc["age"]["mean"]:.1f} vs median={desc["age"]["50%"]:.1f}'
    },
    'salary': {
        'issue': '극단적 이상치',
        'detail': f'min={desc["salary"]["min"]:.0f}\nmax={desc["salary"]["max"]:,.0f}\n연봉 500만??',
        'color': C['weight'],
        'mean_median': f'mean={desc["salary"]["mean"]:,.0f} vs median={desc["salary"]["50%"]:,.0f}'
    },
    'score': {
        'issue': '결측값(Missing)',
        'detail': f'count={desc["score"]["count"]:.0f} / 200\n5개 누락!',
        'color': C['bias'],
        'mean_median': f'count가 200이 아님!'
    },
}

for idx, (col, info) in enumerate(problems_found.items()):
    ax = axes[idx]
    ax.set_facecolor(C['card'])

    data = df_problem[col].dropna()
    ax.hist(data, bins=30, color=info['color'], alpha=0.5, edgecolor='white', lw=0.5)
    ax.set_title(f'{col}: {info["issue"]}', color=info['color'],
                 fontsize=13, fontweight='bold')
    ax.text(0.95, 0.95, info['detail'], transform=ax.transAxes,
            ha='right', va='top', color=info['color'], fontsize=10,
            fontweight='bold', fontfamily='monospace',
            bbox=dict(facecolor=C['bg'], edgecolor=info['color'],
                     boxstyle='round,pad=0.3', alpha=0.9))
    ax.text(0.95, 0.65, info['mean_median'], transform=ax.transAxes,
            ha='right', va='top', color=C['accent'], fontsize=9,
            fontfamily='monospace')

    ax.tick_params(colors=C['muted'], labelsize=8)
    for spine in ax.spines.values():
        spine.set_color(C['muted'])
        spine.set_alpha(0.3)

plt.suptitle('describe() 하나로 발견하는 데이터 문제 3가지',
             color=C['text'], fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("🐍 코드 단 한 줄:")
print("   df.describe()")
print()
display(desc.round(1))
print(f"\n🔗 AI 연결:")
print(f"   'Garbage In, Garbage Out' — 나쁜 데이터로 학습한 모델은 나쁜 예측을 한다.")
print(f"   OpenAI도 GPT 학습 전에 수개월간 데이터 정제(Cleaning)에 투자한다.")
print(f"   describe()는 그 첫 번째 단계이다.")


---

## 7. 데이터 타입(dtype) — "왜 float32와 float64가 중요한가?"

NumPy 배열의 dtype은 단순한 기술 설정이 아니다. **딥러닝의 학습 속도와 메모리 사용량**을 결정한다.

| dtype | 바이트 | 정밀도 | 용도 |
|---|:---:|---|---|
| `float64` | 8 | 매우 높음 | 과학 계산, 통계 분석 (NumPy 기본값) |
| `float32` | 4 | 충분 | **딥러닝 학습의 표준** (PyTorch 기본값) |
| `float16` | 2 | 낮음 | 대형 모델 추론 (GPT 서빙) |
| `bfloat16` | 2 | 중간 | **LLM 학습** (Google TPU, A100 GPU) |
| `int8` | 1 | 정수만 | 모델 경량화(Quantization) — 스마트폰 AI |

> **🔗 AI 연결**: GPT-4는 학습 시 bfloat16, 서빙 시 int8 **양자화**(Quantization)를 사용한다. 정밀도를 낮추면 모델 크기가 절반~1/4로 줄어든다.


In [ ]:
# ═══════════════════════════════════════════════════
# dtype별 메모리·속도 비교
# ═══════════════════════════════════════════════════

n = 10_000_000  # 1천만 개

dtypes = ['float64', 'float32', 'float16']
memories = []
times = []

for dt in dtypes:
    arr = np.random.randn(n).astype(dt)
    memories.append(arr.nbytes / 1e6)

    start = time.perf_counter()
    for _ in range(10):
        _ = arr * 2.0
    t = (time.perf_counter() - start) / 10
    times.append(t * 1000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor=C['bg'])

colors = [C['input'], C['weight'], C['act']]

# 메모리
ax1.set_facecolor(C['card'])
bars = ax1.bar(dtypes, memories, color=colors, alpha=0.7, edgecolor='white', lw=0.5)
for bar, mem in zip(bars, memories):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{mem:.0f} MB', ha='center', color=C['text'], fontsize=12,
             fontweight='bold')
ax1.set_title('메모리 사용량 (원소 1천만 개)', color=C['text'],
              fontsize=13, fontweight='bold')
ax1.set_ylabel('MB', color=C['dim'])
ax1.tick_params(colors=C['muted'])
for spine in ax1.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

# 속도
ax2.set_facecolor(C['card'])
bars = ax2.bar(dtypes, times, color=colors, alpha=0.7, edgecolor='white', lw=0.5)
for bar, t in zip(bars, times):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{t:.1f} ms', ha='center', color=C['text'], fontsize=12,
             fontweight='bold')
ax2.set_title('연산 속도 (×2 곱하기)', color=C['text'],
              fontsize=13, fontweight='bold')
ax2.set_ylabel('ms', color=C['dim'])
ax2.tick_params(colors=C['muted'])
for spine in ax2.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 float64 → float32: 메모리 {memories[0]/memories[1]:.0f}배 절약, 속도 {times[0]/times[1]:.1f}배")
print(f"   float64 → float16: 메모리 {memories[0]/memories[2]:.0f}배 절약")
print(f"\n🔗 AI 연결:")
print(f"   GPT-4 (추정 1.8조 파라미터)를 float32로 저장하면 ≈ 7.2 TB")
print(f"   float16으로 줄이면 ≈ 3.6 TB")
print(f"   int8로 양자화하면 ≈ 1.8 TB → 비로소 서빙 가능")
print(f"   → dtype 선택이 곧 '이 모델을 실제로 돌릴 수 있는가?'를 결정한다")


---

## 🎯 핵심 정리 — 4주차에서 배운 "왜?"

| 개념 | 도구 수준 (기존) | 전문가 수준 (보강) | 최신 AI 연결 |
|---|---|---|---|
| **ndarray** | "배열을 만든다" | 연속 메모리 → CPU/GPU 최적화 | PyTorch tensor와 동일 구조 |
| **텐서 차원** | "shape가 있다" | 스칼라→벡터→행렬→텐서 | 이미지(3D), 배치(4D), 시퀀스(3D) |
| **벡터화** | "for문 대신 쓰면 빠르다" | SIMD 병렬 처리의 원리 | GPU 수천 코어 병렬 = 딥러닝의 엔진 |
| **브로드캐스팅** | "크기 다른 배열도 연산 가능" | 메모리 복사 없이 자동 확장 | 편향 덧셈, 배치 정규화, 어텐션 마스크 |
| **DataFrame** | "표를 다룬다" | 특성 행렬(X)과 타겟(y)의 원천 | 모든 ML의 시작점, Feature Engineering |
| **describe()** | "통계를 본다" | 데이터 품질 점검의 첫 단계 | Data-Centric AI 패러다임 |
| **dtype** | "타입을 지정한다" | 메모리·속도·정밀도 트레이드오프 | float16/bfloat16/int8 → LLM 경량화 |

### 핵심 메시지

> *"NumPy는 '편리한 도구'가 아니라 '현대 AI의 기반 기술'이다.*
> *ndarray의 구조를 이해하면 PyTorch를 이해하고,*
> *벡터화를 이해하면 GPU 가속을 이해하고,*
> *DataFrame을 이해하면 특성 공학을 이해한다."*
